In [1]:
import polars as pl
print("import success")

import success


### Read MIND Training Dataset, Split into Train and Validation

In [2]:
behaviors = pl.read_csv(
    "train/behaviors.tsv", 
    separator="\t", 
    has_header=False, 
    new_columns=["session_id", "user_id", "timestamp", "history", "impressions"]
)
print(behaviors.height)
behaviors.head()

2232748


session_id,user_id,timestamp,history,impressions
i64,str,str,str,str
1,"""U87243""","""11/10/2019 11:30:54 AM""","""N8668 N39081 N65259 N79529 N73…","""N78206-0 N26368-0 N7578-0 N585…"
2,"""U598644""","""11/12/2019 1:45:29 PM""","""N56056 N8726 N70353 N67998 N83…","""N47996-0 N82719-0 N117066-0 N8…"
3,"""U532401""","""11/13/2019 11:23:03 AM""","""N128643 N87446 N122948 N9375 N…","""N103852-0 N53474-0 N127836-0 N…"
4,"""U593596""","""11/12/2019 12:24:09 PM""","""N31043 N39592 N4104 N8223 N114…","""N38902-0 N76434-0 N71593-0 N10…"
5,"""U239687""","""11/14/2019 8:03:01 PM""","""N65250 N122359 N71723 N53796 N…","""N76209-0 N48841-0 N67937-0 N62…"


In [3]:
behaviors = behaviors.with_columns(
    pl.col("timestamp").str.to_datetime(format="%m/%d/%Y %I:%M:%S %p")
).sort("timestamp")
behaviors.head()

session_id,user_id,timestamp,history,impressions
i64,str,datetime[μs],str,str
472831,"""U403465""",2019-11-09 00:00:00,"""N59850 N104930 N68866 N82374 N…","""N92613-0 N17456-0 N67369-0 N31…"
526311,"""U493092""",2019-11-09 00:00:02,"""N3960 N115631 N21659 N79594 N1…","""N6865-0 N128031-0 N79990-0 N12…"
709414,"""U172654""",2019-11-09 00:00:03,"""N43253 N119785 N81937 N128965 …","""N65783-1 N119284-0 N76810-0"""
1626676,"""U248125""",2019-11-09 00:00:07,"""N102417 N22902 N95420 N81970 N…","""N90042-0 N2598-0 N108809-0 N10…"
1104144,"""U495159""",2019-11-09 00:00:13,"""N88765 N74317 N63723 N121794 N…","""N104437-0 N76810-0 N98657-0 N2…"


In [4]:
split_index = int(behaviors.height * 0.85)
behaviors_train = behaviors.slice(0, split_index)
behaviors_validation = behaviors.slice(split_index)
print(f"Size of train set: {behaviors_train.height}, from {behaviors_train['timestamp'].min()} to {behaviors_train['timestamp'].max()}")
print(f"Size of validation set: {behaviors_validation.height}, from {behaviors_validation['timestamp'].min()} to {behaviors_validation['timestamp'].max()}")

Size of train set: 1897835, from 2019-11-09 00:00:00 to 2019-11-14 07:30:28
Size of validation set: 334913, from 2019-11-14 07:30:28 to 2019-11-14 23:59:59


In [5]:
behaviors_train.write_parquet("behaviors_train.parquet")
behaviors_validation.write_parquet("behaviors_validation.parquet")

In [6]:
news = pl.read_csv(
    "train/news.tsv", 
    separator="\t",
    quote_char=None,
    has_header=False,
    new_columns=["news_id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"]
)
print(news.height)
news.head()

101527


news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
str,str,str,str,str,str,str,str
"""N88753""","""lifestyle""","""lifestyleroyals""","""The Brands Queen Elizabeth, Pr…","""Shop the notebooks, jackets, a…","""https://assets.msn.com/labs/mi…","""[{""Label"": ""Prince Philip, Duk…","""[]"""
"""N45436""","""news""","""newsscienceandtechnology""","""Walmart Slashes Prices on Last…","""Apple's new iPad releases brin…","""https://assets.msn.com/labs/mi…","""[{""Label"": ""IPad"", ""Type"": ""J""…","""[{""Label"": ""IPad"", ""Type"": ""J""…"
"""N23144""","""health""","""weightloss""","""50 Worst Habits For Belly Fat""","""These seemingly harmless habit…","""https://assets.msn.com/labs/mi…","""[{""Label"": ""Adipose tissue"", ""…","""[{""Label"": ""Adipose tissue"", ""…"
"""N86255""","""health""","""medical""","""Dispose of unwanted prescripti…",null,"""https://assets.msn.com/labs/mi…","""[{""Label"": ""Drug Enforcement A…","""[]"""
"""N93187""","""news""","""newsworld""","""The Cost of Trump's Aid Freeze…","""Lt. Ivan Molchanets peeked ove…","""https://assets.msn.com/labs/mi…","""[]""","""[{""Label"": ""Ukraine"", ""Type"": …"


In [7]:
def get_split_news_ids(df_behaviors: pl.DataFrame) -> set:
    # 1. Extract history IDs
    history_ids = (
        df_behaviors
        .select(
            pl.col("history")
            .drop_nulls()
            .str.split(" ")
            .explode(empty_as_null=True)
        )
        .filter(pl.col("history") != "")
        .rename({"history": "news_id"})
    )

    # 2. Extract impression IDs
    impression_ids = (
        df_behaviors
        .select(
            pl.col("impressions")
            .drop_nulls()
            .str.split(" ")
            .explode(empty_as_null=True)
        )
        .filter(pl.col("impressions") != "")
        .select(pl.col("impressions").str.extract(r"^([^-]+)", 1).alias("news_id"))
    )

    # Return as a set to avoid the is_in() ambiguity warning
    return set(
        pl.concat([history_ids, impression_ids])
        .unique()["news_id"]
        .to_list()
    )

# Extract unique ID sets
train_news_ids = get_split_news_ids(behaviors_train)
validation_news_ids = get_split_news_ids(behaviors_validation)

# Filter news dataframes without warnings
news_train = news.filter(pl.col("news_id").is_in(train_news_ids))
news_validation = news.filter(pl.col("news_id").is_in(validation_news_ids))

print(news_train.height)
print(news_validation.height)

98671
71991


In [8]:
news_train.write_parquet("news_train.parquet")
news_validation.write_parquet("news_validation.parquet")

### Read MIND Validation Dataset, Use as Testing

In [9]:
behaviors_test = pl.read_csv(
    "test/behaviors.tsv", 
    separator="\t", 
    has_header=False, 
    new_columns=["session_id", "user_id", "timestamp", "history", "impressions"]
)
print(behaviors_test.height)
behaviors_test.head()

376471


session_id,user_id,timestamp,history,impressions
i64,str,str,str,str
1,"""U134050""","""11/15/2019 8:55:22 AM""","""N12246 N128820 N119226 N4065 N…","""N91737-0 N30206-0 N54368-0 N11…"
2,"""U254959""","""11/15/2019 11:42:35 AM""","""N34011 N9375 N67397 N7936 N118…","""N119999-0 N24958-0 N104054-0 N…"
3,"""U499841""","""11/15/2019 9:08:21 AM""","""N63858 N26834 N6379 N85484 N15…","""N18190-0 N89764-0 N91737-0 N54…"
4,"""U107107""","""11/15/2019 5:50:31 AM""","""N12959 N8085 N18389 N3758 N974…","""N122944-1 N18190-0 N55801-0 N5…"
5,"""U492344""","""11/15/2019 5:02:25 AM""","""N109183 N48453 N85005 N45706 N…","""N64785-0 N82503-0 N32993-0 N12…"


In [10]:
behaviors_test = behaviors_test.with_columns(
    pl.col("timestamp").str.to_datetime(format="%m/%d/%Y %I:%M:%S %p")
)
behaviors_test.head()

session_id,user_id,timestamp,history,impressions
i64,str,datetime[μs],str,str
1,"""U134050""",2019-11-15 08:55:22,"""N12246 N128820 N119226 N4065 N…","""N91737-0 N30206-0 N54368-0 N11…"
2,"""U254959""",2019-11-15 11:42:35,"""N34011 N9375 N67397 N7936 N118…","""N119999-0 N24958-0 N104054-0 N…"
3,"""U499841""",2019-11-15 09:08:21,"""N63858 N26834 N6379 N85484 N15…","""N18190-0 N89764-0 N91737-0 N54…"
4,"""U107107""",2019-11-15 05:50:31,"""N12959 N8085 N18389 N3758 N974…","""N122944-1 N18190-0 N55801-0 N5…"
5,"""U492344""",2019-11-15 05:02:25,"""N109183 N48453 N85005 N45706 N…","""N64785-0 N82503-0 N32993-0 N12…"


In [12]:
behaviors_test.write_parquet("behaviors_test.parquet")

In [13]:
news_test = pl.read_csv(
    "test/news.tsv", 
    separator="\t",
    quote_char=None,
    has_header=False,
    new_columns=["news_id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"]
)
print(news_test.height)
news_test.head()

72023


news_id,category,subcategory,title,abstract,url,title_entities,abstract_entities
str,str,str,str,str,str,str,str
"""N88753""","""lifestyle""","""lifestyleroyals""","""The Brands Queen Elizabeth, Pr…","""Shop the notebooks, jackets, a…","""https://assets.msn.com/labs/mi…","""[{""Label"": ""Prince Philip, Duk…","""[]"""
"""N23144""","""health""","""weightloss""","""50 Worst Habits For Belly Fat""","""These seemingly harmless habit…","""https://assets.msn.com/labs/mi…","""[{""Label"": ""Adipose tissue"", ""…","""[{""Label"": ""Adipose tissue"", ""…"
"""N86255""","""health""","""medical""","""Dispose of unwanted prescripti…",null,"""https://assets.msn.com/labs/mi…","""[{""Label"": ""Drug Enforcement A…","""[]"""
"""N93187""","""news""","""newsworld""","""The Cost of Trump's Aid Freeze…","""Lt. Ivan Molchanets peeked ove…","""https://assets.msn.com/labs/mi…","""[]""","""[{""Label"": ""Ukraine"", ""Type"": …"
"""N75236""","""health""","""voices""","""I Was An NBA Wife. Here's How …","""I felt like I was a fraud, and…","""https://assets.msn.com/labs/mi…","""[]""","""[{""Label"": ""National Basketbal…"


In [14]:
news_test.write_parquet("news_test.parquet")